In [1]:
import numpy as np

#### Model

In [2]:
p_x_given_y = np.zeros((2, 2))
p_x_given_y[0, 0] = 0.5
p_x_given_y[0, 1] = 0.3
p_x_given_y[1, 0] = 0.5
p_x_given_y[1, 1] = 0.7

p_y_given_a_z = np.zeros((2, 2, 2))
p_y_given_a_z[0, 0, 0] = 0.1
p_y_given_a_z[0, 0, 1] = 0.2
p_y_given_a_z[0, 1, 0] = 0.3
p_y_given_a_z[0, 1, 1] = 0.4
p_y_given_a_z[1, 0, 0] = 0.9
p_y_given_a_z[1, 0, 1] = 0.8
p_y_given_a_z[1, 1, 0] = 0.7
p_y_given_a_z[1, 1, 1] = 0.6

p_a = np.zeros(2)
p_a[0] = 0.8
p_a[1] = 0.2

p_z = np.zeros(2)
p_z[0] = 0.4
p_z[1] = 0.6

#### Analytical

In [3]:
p_x_and_y_and_z_and_a = np.einsum('ij,jkl,l,k->ijlk', p_x_given_y, p_y_given_a_z, p_z, p_a)

print('p(x) = ' + str(np.einsum('ijkl->i', p_x_and_y_and_z_and_a)))
print('p(y) = ' + str(np.einsum('ijkl->j', p_x_and_y_and_z_and_a)))
print('p(z) = ' + str(np.einsum('ijkl->k', p_x_and_y_and_z_and_a)))
print('p(a) = ' + str(np.einsum('ijkl->l', p_x_and_y_and_z_and_a)))

p_x_given_yeq0 = np.einsum('ijk->i', p_x_and_y_and_z_and_a[:, 0]) / np.sum(np.einsum('ijk->i', p_x_and_y_and_z_and_a[:, 0]))
p_z_given_yeq0 = np.einsum('ijk->j', p_x_and_y_and_z_and_a[:, 0]) / np.sum(np.einsum('ijk->j', p_x_and_y_and_z_and_a[:, 0]))
p_a_given_yeq0 = np.einsum('ijk->k', p_x_and_y_and_z_and_a[:, 0]) / np.sum(np.einsum('ijk->k', p_x_and_y_and_z_and_a[:, 0]))

print('p(x|y=0) = ' + str(np.round(p_x_given_yeq0, 4)))
print('p(z|y=0) = ' + str(np.round(p_z_given_yeq0, 4)))
print('p(a|y=0) = ' + str(np.round(p_a_given_yeq0, 4)))

p(x) = [0.34 0.66]
p(y) = [0.2 0.8]
p(z) = [0.4 0.6]
p(a) = [0.8 0.2]
p(x|y=0) = [0.5 0.5]
p(z|y=0) = [0.28 0.72]
p(a|y=0) = [0.64 0.36]


#### $q(x,y,z,a)=\frac{q(x,y)q(y,z,a)}{q(y)}$ **(Sum Product)**

In [30]:
r_mu_x = np.ones(2)
l_mu_z = p_z
u_mu_a = p_a
r_mu_y = np.einsum('ij,i->j', p_x_given_y, r_mu_x)
#r_mu_y = np.array([1, 0])
l_mu_y = np.einsum('ijk,j,k->i', p_y_given_a_z, u_mu_a, l_mu_z)
#l_mu_y = np.array([1, 0])
l_mu_x = np.einsum('ij,j->i', p_x_given_y, l_mu_y)
r_mu_z = np.einsum('ijk,i,j->k', p_y_given_a_z, r_mu_y, u_mu_a)
d_mu_a = np.einsum('ijk,i,k->j', p_y_given_a_z, r_mu_y, l_mu_z)

q_x = (r_mu_x * l_mu_x) / np.sum(r_mu_x * l_mu_x)
q_y = (r_mu_y * l_mu_y) / np.sum(r_mu_y * l_mu_y)
q_z = (r_mu_z * l_mu_z) / np.sum(r_mu_z * l_mu_z)
q_a = (d_mu_a * u_mu_a) / np.sum(d_mu_a * u_mu_a)

print('q(x) = ' + str(np.round(q_x, 4)))
print('q(y) = ' + str(np.round(q_y, 4)))
print('q(z) = ' + str(np.round(q_z, 4)))
print('q(a) = ' + str(np.round(q_a, 4)))


q(x) = [0.34 0.66]
q(y) = [0.2 0.8]
q(z) = [0.4 0.6]
q(a) = [0.8 0.2]


#### $q(x,y,z,a)=q(x)q(y)q(z)q(a)$ **(Variational Message Passing)**

In [34]:
# Initialization

q_x = np.zeros(2)
q_y = np.zeros(2)
q_z = p_z.copy()
q_a = p_a.copy()

In [44]:
# Iterating ...

r_mu_x = np.ones(2)
l_mu_z = p_z
u_mu_a = p_a
r_mu_y = np.exp(np.einsum('i,ij->j', q_x, np.log(p_x_given_y)))
#r_mu_y = np.array([1, 0])
l_mu_y = np.exp(np.einsum('i,j,kji->k', q_z, q_a, np.log(p_y_given_a_z)))
#l_mu_y = np.array([1, 0])
l_mu_x = np.exp(np.einsum('i,ji->j', q_y, np.log(p_x_given_y)))
r_mu_z = np.exp(np.einsum('i,j,ijk->k', q_y, q_a, np.log(p_y_given_a_z)))
d_mu_a = np.exp(np.einsum('i,j,ikj->k', q_y, q_z, np.log(p_y_given_a_z)))

q_x[...] = (r_mu_x * l_mu_x) / np.sum(r_mu_x * l_mu_x)
q_y[...] = (r_mu_y * l_mu_y) / np.sum(r_mu_y * l_mu_y)
q_z[...] = (r_mu_z * l_mu_z) / np.sum(r_mu_z * l_mu_z)
q_a[...] = (d_mu_a * u_mu_a) / np.sum(d_mu_a * u_mu_a)

print('q(x) = ' + str(np.round(q_x, 4)))
print('q(y) = ' + str(np.round(q_y, 4)))
print('q(z) = ' + str(np.round(q_z, 4)))
print('q(a) = ' + str(np.round(q_a, 4)))


q(x) = [0.332 0.668]
q(y) = [0.1747 0.8253]
q(z) = [0.3988 0.6012]
q(a) = [0.8119 0.1881]


#### $q(x,y,z,a)=q(x)q(y)q(a,z)$ **(Structured Variational Message Passing)**

In [46]:
# Initialization

q_x = np.zeros(2)
q_y = np.zeros(2)
q_z = p_z.copy()
q_a = p_a.copy()
q_a_and_z = np.zeros((2, 2))

In [57]:
# Iterating ...

r_mu_x = np.ones(2)
l_mu_z = p_z
u_mu_a = p_a
r_mu_y = np.exp(np.einsum('i,ij->j', q_x, np.log(p_x_given_y)))
#r_mu_y = np.array([1, 0])
l_mu_y = np.exp(np.einsum('ij,kij->k', q_a_and_z, np.log(p_y_given_a_z)))
#l_mu_y = np.array([1, 0])
l_mu_x = np.exp(np.einsum('i,ji->j', q_y, np.log(p_x_given_y)))
f_a_and_z = np.exp(np.einsum('i,ijk->jk', q_y, np.log(p_y_given_a_z)))
r_mu_z = np.einsum('ij,i->j', f_a_and_z, u_mu_a)
d_mu_a = np.einsum('ij,j->i', f_a_and_z, l_mu_z)

q_x[...] = (r_mu_x * l_mu_x) / np.sum(r_mu_x * l_mu_x)
q_y[...] = (r_mu_y * l_mu_y) / np.sum(r_mu_y * l_mu_y)
q_z[...] = (r_mu_z * l_mu_z) / np.sum(r_mu_z * l_mu_z)
q_a[...] = (d_mu_a * u_mu_a) / np.sum(d_mu_a * u_mu_a)
q_a_and_z[...] = (f_a_and_z * l_mu_z * u_mu_a) / np.sum(f_a_and_z * l_mu_z * u_mu_a)

print('q(x) = ' + str(np.round(q_x, 4)))
print('q(y) = ' + str(np.round(q_y, 4)))
print('q(z) = ' + str(np.round(q_z, 4)))
print('q(a) = ' + str(np.round(q_a, 4)))


q(x) = [0.3361 0.6639]
q(y) = [0.1967 0.8033]
q(z) = [0.3951 0.6049]
q(a) = [0.808 0.192]


#### $q(x,y,z,a)=q(x)q(y,z)q(a)$ **(Structured Variational Message Passing)**

In [58]:
# Initialization ...

q_x = np.zeros(2)
q_y = np.zeros(2)
q_z = p_z.copy()
q_a = p_a.copy()
q_y_and_z = np.zeros((2, 2))

In [66]:
# Iterating ...

r_mu_x = np.ones(2)
l_mu_z = p_z
u_mu_a = p_a
r_mu_y = np.exp(np.einsum('i,ij->j', q_x, np.log(p_x_given_y)))
#r_mu_y = np.array([1, 0])
l_mu_x = np.exp(np.einsum('i,ji->j', q_y, np.log(p_x_given_y)))
f_y_and_z = np.exp(np.einsum('i,jik->jk', q_a, np.log(p_y_given_a_z)))
l_mu_y = np.einsum('ij,j->i', f_y_and_z, l_mu_z)
#l_mu_y = np.array([1, 0])
r_mu_z = np.einsum('ij,i->j', f_y_and_z, r_mu_y)
d_mu_a = np.exp(np.einsum('ij,ikj->k', q_y_and_z, np.log(p_y_given_a_z)))

q_x[...] = (r_mu_x * l_mu_x) / np.sum(r_mu_x * l_mu_x)
q_y[...] = (r_mu_y * l_mu_y) / np.sum(r_mu_y * l_mu_y)
q_z[...] = (r_mu_z * l_mu_z) / np.sum(r_mu_z * l_mu_z)
q_a[...] = (d_mu_a * u_mu_a) / np.sum(d_mu_a * u_mu_a)
q_y_and_z[...] = (f_y_and_z * r_mu_y * l_mu_z) / np.sum(f_y_and_z * r_mu_y * l_mu_z)

print('q(x=0) = ' + str(np.round(q_x, 4)))
print('q(y=0) = ' + str(np.round(q_y, 4)))
print('q(z=0) = ' + str(np.round(q_z, 4)))
print('q(a=0) = ' + str(np.round(q_a, 4)))

q(x=0) = [0.3331 0.6669]
q(y=0) = [0.1809 0.8191]
q(z=0) = [0.4003 0.5997]
q(a=0) = [0.8108 0.1892]
